In [1]:
import pandas as pd
import numpy as np
import difflib

In [2]:
credit=pd.read_csv('tmdb_5000_credits.csv')
movie=pd.read_csv('tmdb_5000_movies.csv')

In [3]:
print(credit.columns)
print(movie.columns)

Index(['movie_id', 'title', 'cast', 'crew'], dtype='object')
Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='object')


In [4]:
print(credit.head())
print(movie.head())

   movie_id                                     title  \
0     19995                                    Avatar   
1       285  Pirates of the Caribbean: At World's End   
2    206647                                   Spectre   
3     49026                     The Dark Knight Rises   
4     49529                               John Carter   

                                                cast  \
0  [{"cast_id": 242, "character": "Jake Sully", "...   
1  [{"cast_id": 4, "character": "Captain Jack Spa...   
2  [{"cast_id": 1, "character": "James Bond", "cr...   
3  [{"cast_id": 2, "character": "Bruce Wayne / Ba...   
4  [{"cast_id": 5, "character": "John Carter", "c...   

                                                crew  
0  [{"credit_id": "52fe48009251416c750aca23", "de...  
1  [{"credit_id": "52fe4232c3a36847f800b579", "de...  
2  [{"credit_id": "54805967c3a36829b5002c41", "de...  
3  [{"credit_id": "52fe4781c3a36847f81398c3", "de...  
4  [{"credit_id": "52fe479ac3a36847f813eaa3",

In [5]:
credit=credit[['title', 'cast', 'crew']]
movie=movie[['id', 'title','overview','genres','keywords']]

In [6]:
print(credit.columns)
print(movie.columns)

Index(['title', 'cast', 'crew'], dtype='object')
Index(['id', 'title', 'overview', 'genres', 'keywords'], dtype='object')


In [7]:
print(credit.head(1))
print(movie.head(1))

    title                                               cast  \
0  Avatar  [{"cast_id": 242, "character": "Jake Sully", "...   

                                                crew  
0  [{"credit_id": "52fe48009251416c750aca23", "de...  
      id   title                                           overview  \
0  19995  Avatar  In the 22nd century, a paraplegic Marine is di...   

                                              genres  \
0  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   

                                            keywords  
0  [{"id": 1463, "name": "culture clash"}, {"id":...  


In [8]:
#Left join returns all rows from the left DataFrame and only the matching rows from the right DataFrame. If there is no match, the right-side columns contain NaN
movie = movie.merge(credit,on='title',how='left') 

In [9]:
movie.head(1)

,id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [10]:
movie.isnull().sum()

id          0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [11]:
movie.dropna(inplace=True)

In [12]:
movie.isnull().sum()

id          0
title       0
overview    0
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [13]:
movie.iloc[0].genres

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [14]:
for i in movie.columns:
    print(f"{i}: {type(movie[i].iloc[0])}")

id: <class 'numpy.int64'>
title: <class 'str'>
overview: <class 'str'>
genres: <class 'str'>
keywords: <class 'str'>
cast: <class 'str'>
crew: <class 'str'>


In [15]:
import ast

In [17]:
def convert(obj):
    L=[]
    #ast.literal_eval is used to convert string to list of dictionaries
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

In [18]:
movie['genres']=movie['genres'].apply(convert)
movie['keywords']=movie['keywords'].apply(convert)
movie['cast']=movie['cast'].apply(convert)
# movie['crew']=movie['crew'].apply(convert)    Not doing because we are choosing only job == director role

In [19]:
for i in movie.columns:
    print(f"{i}: {type(movie[i].iloc[0])}")

id: <class 'numpy.int64'>
title: <class 'str'>
overview: <class 'str'>
genres: <class 'list'>
keywords: <class 'list'>
cast: <class 'list'>
crew: <class 'str'>


In [20]:
def convert3(obj):
    L=[]
    counter=0
    # Here we not used ast.literal_eval because already cast is in list.
    for i in obj:
        if counter<3:
            L.append(i)
            counter+=1
        else:
            break
    return L

In [21]:
movie['cast']=movie['cast'].apply(convert3)

In [22]:
# Here we are fetching only the directors 
def fetch(obj):
    L=[]
    for i in ast.literal_eval(obj):
        if i['job']=='Director':
            L.append(i['name'])
            break
    return L

In [23]:
movie['crew']=movie['crew'].apply(fetch)

In [24]:
movie.head()

,id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes]
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman]",[Christopher Nolan]
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton]",[Andrew Stanton]


In [25]:
movie['overview'][0]

'In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.'

In [26]:
# Converts to list so that we can combine easily with other lists
movie['overview']=movie['overview'].apply(lambda x:x.split()) 

In [27]:
# Removing spaces in genres,keywords,cast,crew
movie['genres']=movie['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movie['keywords']=movie['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movie['cast']=movie['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movie['crew']=movie['crew'].apply(lambda x:[i.replace(" ","") for i in x])

In [28]:
movie.head()

,id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron]
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski]
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...","[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes]
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dccomics, crimefighter, terrorist, secretiden...","[ChristianBale, MichaelCaine, GaryOldman]",[ChristopherNolan]
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, ScienceFiction]","[basedonnovel, mars, medallion, spacetravel, p...","[TaylorKitsch, LynnCollins, SamanthaMorton]",[AndrewStanton]


In [29]:
# Mergin all data and keep them in another column
movie['tag']=movie['overview']+movie['genres']+movie['keywords']+movie['cast']+movie['crew']

In [30]:
movie.head()

,id,title,overview,genres,keywords,cast,crew,tag
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron],"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski],"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...","[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes],"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dccomics, crimefighter, terrorist, secretiden...","[ChristianBale, MichaelCaine, GaryOldman]",[ChristopherNolan],"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, ScienceFiction]","[basedonnovel, mars, medallion, spacetravel, p...","[TaylorKitsch, LynnCollins, SamanthaMorton]",[AndrewStanton],"[John, Carter, is, a, war-weary,, former, mili..."


In [31]:
movies=movie[['id','title','tag']]

In [32]:
movies.head()

,id,title,tag
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili..."


In [33]:
movies['tag'][0]

['In',
 'the',
 '22nd',
 'century,',
 'a',
 'paraplegic',
 'Marine',
 'is',
 'dispatched',
 'to',
 'the',
 'moon',
 'Pandora',
 'on',
 'a',
 'unique',
 'mission,',
 'but',
 'becomes',
 'torn',
 'between',
 'following',
 'orders',
 'and',
 'protecting',
 'an',
 'alien',
 'civilization.',
 'Action',
 'Adventure',
 'Fantasy',
 'ScienceFiction',
 'cultureclash',
 'future',
 'spacewar',
 'spacecolony',
 'society',
 'spacetravel',
 'futuristic',
 'romance',
 'space',
 'alien',
 'tribe',
 'alienplanet',
 'cgi',
 'marine',
 'soldier',
 'battle',
 'loveaffair',
 'antiwar',
 'powerrelations',
 'mindandsoul',
 '3d',
 'SamWorthington',
 'ZoeSaldana',
 'SigourneyWeaver',
 'JamesCameron']

In [34]:
movies['tag']=movies['tag'].apply(lambda x:' '.join(x))

C:\Users\newse\AppData\Local\Temp\ipykernel_15532\2395894393.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies['tag']=movies['tag'].apply(lambda x:' '.join(x))


In [35]:
movies.head()

,id,title,tag
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...
4,49529,John Carter,"John Carter is a war-weary, former military ca..."


In [36]:
movies['tag'][0]

'In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. Action Adventure Fantasy ScienceFiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d SamWorthington ZoeSaldana SigourneyWeaver JamesCameron'

In [37]:
movies['tag']=movies['tag'].apply(lambda x:x.lower())

C:\Users\newse\AppData\Local\Temp\ipykernel_15532\2923874054.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies['tag']=movies['tag'].apply(lambda x:x.lower())


In [38]:
movies['tag'][0]

'in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver jamescameron'

In [39]:
# Helps to convert the string in tags to numbers
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer(max_features=5000,stop_words='english')  #Stop_wprds='english' because many words like the,a,an etc.. are removed automatically

In [40]:
cv.fit_transform(movies['tag']).toarray().shape

(4806, 5000)

In [41]:
cv.get_feature_names_out()[:20]

array(['000', '007', '10', '100', '11', '12', '13', '14', '15', '16',
       '17', '18', '18th', '19', '1930s', '1940s', '1950', '1950s',
       '1960s', '1970s'], dtype=object)

In [42]:
vectors=cv.fit_transform(movies['tag']).toarray()

In [43]:
vectors[0]

array([0, 0, 0, ..., 0, 0, 0], dtype=int64)

In [44]:
len(cv.get_feature_names_out())

5000

In [45]:
cv.get_feature_names_out()[:20]

array(['000', '007', '10', '100', '11', '12', '13', '14', '15', '16',
       '17', '18', '18th', '19', '1930s', '1940s', '1950', '1950s',
       '1960s', '1970s'], dtype=object)

In [46]:
import nltk
# NLTK (Natural Language Toolkit) is a Python library for Natural Language Processing (NLP).
# It provides tools for text preprocessing such as tokenization, stemming(Stemming reduces a word to its root (stem) by removing prefixes or suffixes.)
# lemmatization, stop word removal, POS tagging, parsing, classification,and semantic analysis.

In [47]:
from nltk.stem.porter import PorterStemmer #PorterStemmer is an algorithm in NLTK that reduces a word to its root form (stem) by removing common prefixes and suffixes.
ps=PorterStemmer()

In [48]:
def stem(text):
    L=[]
    for i in text.split():
        L.append(ps.stem(i))
    return " ".join(L)

In [49]:
movies['tag']=movies['tag'].apply(stem)

C:\Users\newse\AppData\Local\Temp\ipykernel_15532\388885283.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies['tag']=movies['tag'].apply(stem)


In [50]:
from sklearn.metrics.pairwise import cosine_similarity # Majors the similarity between 2 vectors

In [51]:
cosine_similarity(vectors)

array([[1.        , 0.08964215, 0.05976143, ..., 0.02519763, 0.02817181,
        0.        ],
       [0.08964215, 1.        , 0.0625    , ..., 0.02635231, 0.        ,
        0.        ],
       [0.05976143, 0.0625    , 1.        , ..., 0.02635231, 0.        ,
        0.        ],
       ...,
       [0.02519763, 0.02635231, 0.02635231, ..., 1.        , 0.0745356 ,
        0.04836508],
       [0.02817181, 0.        , 0.        , ..., 0.0745356 , 1.        ,
        0.05407381],
       [0.        , 0.        , 0.        , ..., 0.04836508, 0.05407381,
        1.        ]])

In [52]:
cosine_similarity(vectors).shape

(4806, 4806)

In [53]:
sim=cosine_similarity(vectors)

In [54]:
sim[0]

array([1.        , 0.08964215, 0.05976143, ..., 0.02519763, 0.02817181,
       0.        ])

In [55]:
sorted(list(enumerate(sim[0])),reverse=True, key=lambda x:x[1])[1:6]

[(539, 0.26089696604360174),
 (1192, 0.2581988897471611),
 (507, 0.25302403842552984),
 (260, 0.25110592822973776),
 (1214, 0.24944382578492943)]

In [56]:
def recommend(movie):
    result = movies[movies['title'] == movie]

    if result.empty:
        print("Movie not found")
        return

    movie_idx = result.index[0]
    dist = sim[movie_idx]
    movie_list = sorted(list(enumerate(dist)), reverse=True, key=lambda x: x[1])[1:10]

    for i in movie_list:
        print(movies.iloc[i[0]]['title'])

In [57]:
recommend('Avatar')

Titan A.E.
Small Soldiers
Independence Day
Ender's Game
Aliens vs Predator: Requiem
Battle: Los Angeles
Lifeforce
Falcon Rising
Krull


In [58]:
recommend('Iron Man')

Iron Man 2
Iron Man 3
Avengers: Age of Ultron
Captain America: Civil War
The Avengers
Ant-Man
X-Men
X-Men: The Last Stand
Thor: The Dark World


In [59]:
recommend('Avenger')

Movie not found


In [60]:
from difflib import get_close_matches
def recommend(movie):
    movie_titles = movies['title'].tolist()
    match = get_close_matches(movie, movie_titles, n=1, cutoff=0.5)
    if not match:
        print("Movie not found")
        return
    movie = match[0]      # Closest matching title
    movie_idx = movies[movies['title'] == movie].index[0]
    dist = sim[movie_idx]
    movie_list = sorted(list(enumerate(dist)), reverse=True, key=lambda x: x[1])[1:6]
    print("Showing recommendations for:", movie)
    for i in movie_list:
        print(movies.iloc[i[0]]['title'])

In [61]:
recommend('Avenger')

Showing recommendations for: The Avengers
Avengers: Age of Ultron
Captain America: Civil War
Iron Man 3
Captain America: The First Avenger
Iron Man


In [62]:
import pickle
# Save movies DataFrame
pickle.dump(movies, open("movies.pkl", "wb"))
# Save similarity matrix
pickle.dump(simi, open("similarity.pkl", "wb"))
print("Files saved successfully!")

Files saved successfully!
